In [2]:
# =============================================================================
# DEFINITIVE BASELINE REPLICATION SCRIPT
#
# VERSION: 4.0 (Perfected Replication from Source Notebook)
#
# Author: Critical & Independent Scientific Expert
#
# Description:
# This script is a faithful Python implementation of the methodology found in
# the "Baselines for Mortality and LOS prediction - Sklearn.ipynb" notebook
# provided by the author of Gao et al. (2024). It correctly replicates the
# data splits, the complex feature engineering (normalization, imputation with
# masking, and time-series flattening), and the model selection process
# (hyperparameter search). This script produces a scientifically valid baseline.
# =============================================================================

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score
import os
import sys
import random

print("Libraries imported successfully.")

# =============================================================================
# Section 1: Experiment Configuration (Replicated from Notebook)
# =============================================================================

# -- SAMPLING CONFIGURATION --
# Set USE_SAMPLE to True to sample only N_SAMPLE_PATIENTS for testing/debugging
USE_SAMPLE = False  # Set to True to enable sampling
N_SAMPLE_PATIENTS = 1000  # Number of patients to sample when USE_SAMPLE=True

# -- File Path --
HDF_FILE_PATH = '../../data/raw/all_hourly_data.h5'

# -- Prediction Target --
TARGET_VARIABLE = 'mort_hosp'

## CRITICAL REPLICATION STEP: Constants taken directly from the source notebook.
GAP_TIME = 6
WINDOW_SIZE = 24
SEED = 1
ID_COLS = ['subject_id', 'hadm_id', 'icustay_id']

# Set seeds for reproducibility
np.random.seed(SEED)
random.seed(SEED)

if USE_SAMPLE:
    print(f"⚠ SAMPLING MODE ENABLED: Will use only {N_SAMPLE_PATIENTS} random patients for testing.")
else:
    print("Using full dataset.")
print("Configuration set to match the source notebook.")


Libraries imported successfully.
Configuration set to match the source notebook.


In [ ]:

# =============================================================================
# Section 2: Data Loading & Initial Setup
# =============================================================================

if not os.path.exists(HDF_FILE_PATH):
    sys.exit(f"--- CRITICAL ERROR: File not found at '{HDF_FILE_PATH}' ---")

print("\nLoading data from HDF5 store...")
with pd.HDFStore(HDF_FILE_PATH, 'r') as store:
    df_statics = store.select('/patients')
    # The notebook uses the '/vitals_labs' table with MultiIndex columns (mean, count, std)
    df_ts_full = store.select('/vitals_labs')

print("✓ Data loaded.")

# =============================================================================
# Section 3: Cohort Definition and Data Splitting (Replicated from Notebook)
# =============================================================================

print("\nDefining patient cohort and splitting data...")

## CRITICAL FIX: Implementing the correct, multi-step label creation logic.
# 1. Select the cohort and base labels, including the raw 'los_icu' column.
Ys_full = df_statics[df_statics.max_hours > WINDOW_SIZE + GAP_TIME][['mort_hosp', 'mort_icu', 'los_icu']]

# 2. Apply sampling if enabled
if USE_SAMPLE and len(Ys_full) > N_SAMPLE_PATIENTS:
    print(f"  Sampling {N_SAMPLE_PATIENTS} patients from {len(Ys_full)} eligible patients...")
    # Sample by subject_id to ensure we get complete patient records
    available_subjects = Ys_full.index.get_level_values('subject_id').unique()
    sampled_subjects = np.random.choice(available_subjects, size=min(N_SAMPLE_PATIENTS, len(available_subjects)), replace=False)
    Ys = Ys_full[Ys_full.index.get_level_values('subject_id').isin(sampled_subjects)]
    print(f"  ✓ Sampled {len(Ys)} patient records from {len(sampled_subjects)} subjects")
else:
    Ys = Ys_full
    if USE_SAMPLE:
        print(f"  Note: Requested {N_SAMPLE_PATIENTS} patients, but only {len(Ys_full)} available. Using all available patients.")

# 3. Create the derived boolean LOS labels from 'los_icu'.
Ys['los_3'] = (Ys['los_icu'] > 3).astype(float)
Ys['los_7'] = (Ys['los_icu'] > 7).astype(float)

# 4. Drop the intermediate 'los_icu' column as it is no longer needed.
Ys.drop(columns=['los_icu'], inplace=True)

# 5. Ensure all final label columns are of float type.
Ys = Ys.astype(float)

print(f"✓ Cohort and labels defined correctly. {len(Ys)} patients in final cohort.")

# Filter time-series data for the cohort and time window
df_ts = df_ts_full[
    (df_ts_full.index.get_level_values('icustay_id').isin(set(Ys.index.get_level_values('icustay_id')))) &
    (df_ts_full.index.get_level_values('hours_in') < WINDOW_SIZE)
]

# Replicate the exact 70/10/20 train/dev/test split by subject
all_subjects = np.random.permutation(list(set(Ys.index.get_level_values('subject_id'))))
n_total = len(all_subjects)
n_train = int(n_total * 0.7)
n_dev = int(n_total * 0.1)

train_subj = all_subjects[:n_train]
dev_subj = all_subjects[n_train : n_train + n_dev]
test_subj = all_subjects[n_train + n_dev :]

def split_dataframe_by_subjects(df, subjects_list):
    return df[df.index.get_level_values('subject_id').isin(subjects_list)]

(ts_train, ts_dev, ts_test) = [split_dataframe_by_subjects(df_ts, s) for s in (train_subj, dev_subj, test_subj)]
(Ys_train, Ys_dev, Ys_test) = [split_dataframe_by_subjects(Ys, s) for s in (train_subj, dev_subj, test_subj)]

print("✓ Data split into train/dev/test sets according to notebook logic (70/10/20).")
print(f"Train patients: {len(train_subj)}, Dev patients: {len(dev_subj)}, Test patients: {len(test_subj)}")

In [3]:
# =============================================================================
# Section 2.5: Check for and Load Existing Preprocessed Data
# =============================================================================

import pickle
import glob
from datetime import datetime

print("\n--- Checking for Existing Preprocessed Data ---")

# Check for existing preprocessed data
existing_experiments = glob.glob("../results/baseline_experiment_*/preprocessed_data/X_train.pkl")

load_existing = False
skip_processing = False

if existing_experiments:
    print(f"Found {len(existing_experiments)} existing experiment(s):")
    for i, exp_path in enumerate(existing_experiments, 1):
        exp_dir = os.path.dirname(os.path.dirname(exp_path))
        exp_name = os.path.basename(exp_dir)
        mod_time = datetime.fromtimestamp(os.path.getctime(exp_path)).strftime('%Y-%m-%d %H:%M:%S')
        print(f"  {i}. {exp_name} (created: {mod_time})")
    
    # Use the most recent experiment
    most_recent = max(existing_experiments, key=os.path.getctime)
    most_recent_dir = os.path.dirname(most_recent)
    most_recent_exp = os.path.dirname(most_recent_dir)
    
    print(f"\nMost recent experiment: {os.path.basename(most_recent_exp)}")
    
    # Check if all required files exist
    required_files = [
        "X_train.pkl", "X_dev.pkl", "X_test.pkl",
        "Ys_train.pkl", "Ys_dev.pkl", "Ys_test.pkl"
    ]
    
    all_files_exist = all(os.path.exists(os.path.join(most_recent_dir, f)) for f in required_files)
    
    if all_files_exist:
        print("✓ All required preprocessed files found. Loading existing data...")
        
        # Load preprocessed data
        X_train = pd.read_pickle(os.path.join(most_recent_dir, "X_train.pkl"))
        X_dev = pd.read_pickle(os.path.join(most_recent_dir, "X_dev.pkl"))
        X_test = pd.read_pickle(os.path.join(most_recent_dir, "X_test.pkl"))
        
        Ys_train = pd.read_pickle(os.path.join(most_recent_dir, "Ys_train.pkl"))
        Ys_dev = pd.read_pickle(os.path.join(most_recent_dir, "Ys_dev.pkl"))
        Ys_test = pd.read_pickle(os.path.join(most_recent_dir, "Ys_test.pkl"))
        
        # Also load the split subject lists if available
        try:
            train_subj = X_train.index.get_level_values('subject_id').unique()
            dev_subj = X_dev.index.get_level_values('subject_id').unique()  
            test_subj = X_test.index.get_level_values('subject_id').unique()
        except:
            # If there's any issue, we'll use the newly generated splits
            pass
        
        # Load configuration to verify compatibility
        config_path = os.path.join(most_recent_exp, "experiment_config.pkl")
        if os.path.exists(config_path):
            with open(config_path, 'rb') as f:
                loaded_config = pickle.load(f)
            
            # Check if configuration matches
            config_matches = (
                loaded_config.get('target_variable') == TARGET_VARIABLE and
                loaded_config.get('gap_time') == GAP_TIME and
                loaded_config.get('window_size') == WINDOW_SIZE and
                loaded_config.get('seed') == SEED
            )
            
            if config_matches:
                print(f"✓ Configuration matches current experiment settings")
                load_existing = True
                skip_processing = True
            else:
                print("⚠ Configuration differs from current settings:")
                print(f"  Target: {loaded_config.get('target_variable')} vs {TARGET_VARIABLE}")
                print(f"  Gap time: {loaded_config.get('gap_time')} vs {GAP_TIME}")
                print(f"  Window: {loaded_config.get('window_size')} vs {WINDOW_SIZE}")
                print(f"  Seed: {loaded_config.get('seed')} vs {SEED}")
                print("  Will proceed with fresh preprocessing to ensure consistency.")
        
        if load_existing and skip_processing:
            print(f"✓ Successfully loaded preprocessed data!")
            print(f"  Train: {X_train.shape}")
            print(f"  Dev:   {X_dev.shape}")
            print(f"  Test:  {X_test.shape}")
            print(f"  Skipping feature engineering pipeline...")
        
    else:
        print("⚠ Some required files missing. Will proceed with preprocessing.")
else:
    print("No existing preprocessed data found. Will proceed with fresh preprocessing.")

if not load_existing:
    print("Will run feature engineering pipeline...")



--- Checking for Existing Preprocessed Data ---
Found 1 existing experiment(s):
  1. baseline_experiment_20250621_165948 (created: 2025-06-21 16:59:54)

Most recent experiment: baseline_experiment_20250621_165948
✓ All required preprocessed files found. Loading existing data...
✓ Configuration matches current experiment settings
✓ Successfully loaded preprocessed data!
  Train: (16760, 7488)
  Dev:   (2394, 7488)
  Test:  (4790, 7488)
  Skipping feature engineering pipeline...


In [4]:
# =============================================================================
# Section 4: Feature Engineering Pipeline (Corrected for ValueError)
# =============================================================================

def replicate_feature_pipeline(ts_train, ts_dev, ts_test):
    """
    Applies the full feature engineering pipeline from the source notebook,
    with a robust fix for the pandas column alignment ValueError.
    """
    print("\nStarting full feature engineering replication...")
    idx = pd.IndexSlice

    # 1. Z-Score Normalization
    print("  1. Calculating normalization stats from training data...")
    train_means = ts_train.loc[:, idx[:, 'mean']].mean(axis=0)
    train_stds = ts_train.loc[:, idx[:, 'mean']].std(axis=0).replace(0, 1)

    print("  2. Applying Z-score normalization...")
    for df in [ts_train, ts_dev, ts_test]:
        # Use .loc to avoid SettingWithCopyWarning
        df.loc[:, idx[:, 'mean']] = (df.loc[:, idx[:, 'mean']] - train_means) / train_stds

    # 2. Custom Imputation Function (`simple_imputer` from notebook)
    def custom_imputer(df):
        df_impute = df.copy()
        df_out = df_impute.loc[:, idx[:, ['mean', 'count']]]
        patient_means = df_out.loc[:, idx[:, 'mean']].groupby(ID_COLS).transform('mean')
        df_out.loc[:, idx[:, 'mean']] = df_out.loc[:, idx[:, 'mean']].groupby(ID_COLS).ffill()
        df_out.loc[:, idx[:, 'mean']] = df_out.loc[:, idx[:, 'mean']].fillna(patient_means).fillna(0)
        df_out.loc[:, idx[:, 'count']] = (df_impute.loc[:, idx[:, 'count']] > 0).astype(float)
        df_out = df_out.rename(columns={'count': 'mask'}, level=1)
        
        ## CRITICAL FIX (Version 5.2): Robustly calculate 'time_since_measured'
        # The original code fails in modern pandas because filtering can change the column set.
        
        # Calculate cumulative hours of absence for all columns
        is_absent = 1 - df_out.loc[:, idx[:, 'mask']]
        hours_of_absence = is_absent.groupby(level=ID_COLS).cumsum()
        
        # Create a sparse DataFrame containing the cumulative absence count
        # ONLY at times when a measurement was actually present.
        last_known_absence = hours_of_absence[~is_absent.astype(bool)]
        
        # Forward-fill these last known values.
        ffilled_absence = last_known_absence.groupby(level=ID_COLS).ffill()
        
        # *** THE DEFINITIVE FIX IS HERE ***
        # Reindex the forward-filled data to have the EXACT same columns as the
        # full `hours_of_absence` DataFrame. This prevents the ValueError by
        # ensuring the two DataFrames are perfectly alignable.
        ffilled_absence_aligned = ffilled_absence.reindex(columns=hours_of_absence.columns)

        # Now, the subtraction is safe.
        time_since = hours_of_absence - ffilled_absence_aligned
        time_since = time_since.rename(columns={'mask': 'time_since_measured'}, level=1)
        
        # Combine all engineered features
        df_final = pd.concat([df_out, time_since], axis=1)
        df_final.loc[:, idx[:, 'time_since_measured']] = df_final.loc[:, idx[:, 'time_since_measured']].fillna(100)
        df_final = df_final.sort_index(axis=1)
        return df_final

    print("  3. Applying custom imputation (mean, mask, time_since_measured)...")
    ts_train_imp, ts_dev_imp, ts_test_imp = [custom_imputer(df) for df in [ts_train, ts_dev, ts_test]]

    # 4. Flatten Time-Series via Pivot
    print("  4. Flattening time-series data into wide format...")
    def flatten_features(df):
        return df.pivot_table(index=ID_COLS, columns=['hours_in'])

    X_train_flat = flatten_features(ts_train_imp)
    X_dev_flat = flatten_features(ts_dev_imp)
    X_test_flat = flatten_features(ts_test_imp)

    # Align columns across all sets to handle any discrepancies
    all_cols = X_train_flat.columns.union(X_dev_flat.columns).union(X_test_flat.columns)
    X_train_flat = X_train_flat.reindex(columns=all_cols, fill_value=0)
    X_dev_flat = X_dev_flat.reindex(columns=all_cols, fill_value=0)
    X_test_flat = X_test_flat.reindex(columns=all_cols, fill_value=0)
    
    print("✓ Feature engineering complete.")
    return X_train_flat, X_dev_flat, X_test_flat

# Execute the pipeline only if we didn't load existing data
if not skip_processing:
    X_train, X_dev, X_test = replicate_feature_pipeline(ts_train, ts_dev, ts_test)
else:
    print("\n✓ Using loaded preprocessed data, skipping feature engineering pipeline")

print(f"\nFinal feature matrix shapes:")
print(f"  Train: {X_train.shape}")
print(f"  Dev:   {X_dev.shape}")
print(f"  Test:  {X_test.shape}")




✓ Using loaded preprocessed data, skipping feature engineering pipeline

Final feature matrix shapes:
  Train: (16760, 7488)
  Dev:   (2394, 7488)
  Test:  (4790, 7488)


In [5]:

# =============================================================================
# Section 4.5: Save Preprocessed Data for Future Use (Only if freshly processed)
# =============================================================================

# Only save if we actually processed the data (not loaded existing)
if not skip_processing:
    # Import additional modules if not already imported
    if 'pickle' not in globals():
        import pickle
    if 'datetime' not in globals():
        from datetime import datetime

    # Create output directories
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"../results/baseline_experiment_{timestamp}"
    data_dir = os.path.join(output_dir, "preprocessed_data")
    models_dir = os.path.join(output_dir, "models")
    figures_dir = os.path.join(output_dir, "figures")
    logs_dir = os.path.join(output_dir, "logs")

    for dir_path in [output_dir, data_dir, models_dir, figures_dir, logs_dir]:
        os.makedirs(dir_path, exist_ok=True)

    print(f"\nSaving freshly preprocessed data to: {data_dir}")

    # Save feature matrices
    X_train.to_pickle(os.path.join(data_dir, "X_train.pkl"))
    X_dev.to_pickle(os.path.join(data_dir, "X_dev.pkl"))
    X_test.to_pickle(os.path.join(data_dir, "X_test.pkl"))

    # Save target labels  
    Ys_train.to_pickle(os.path.join(data_dir, "Ys_train.pkl"))
    Ys_dev.to_pickle(os.path.join(data_dir, "Ys_dev.pkl"))
    Ys_test.to_pickle(os.path.join(data_dir, "Ys_test.pkl"))

    # Save raw time series data for reference (only if we have them)
    if 'ts_train' in globals():
        ts_train.to_pickle(os.path.join(data_dir, "ts_train_raw.pkl"))
        ts_dev.to_pickle(os.path.join(data_dir, "ts_dev_raw.pkl"))
        ts_test.to_pickle(os.path.join(data_dir, "ts_test_raw.pkl"))

    # Save experiment configuration
    config = {
        'target_variable': TARGET_VARIABLE,
        'gap_time': GAP_TIME,
        'window_size': WINDOW_SIZE,
        'seed': SEED,
        'id_cols': ID_COLS,
        'use_sample': USE_SAMPLE,
        'n_sample_patients': N_SAMPLE_PATIENTS,
        'train_subjects': len(train_subj),
        'dev_subjects': len(dev_subj),
        'test_subjects': len(test_subj),
        'feature_shapes': {
            'train': X_train.shape,
            'dev': X_dev.shape,
            'test': X_test.shape
        }
    }

    with open(os.path.join(output_dir, "experiment_config.pkl"), "wb") as f:
        pickle.dump(config, f)

    print("✓ Preprocessed data saved successfully")
    print(f"  - Feature matrices: {data_dir}")
    print(f"  - Labels: {data_dir}")
    print(f"  - Configuration: {output_dir}/experiment_config.pkl")
    
else:
    # We loaded existing data, so we need to create a new output directory for this model run
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"../results/baseline_experiment_{timestamp}"
    data_dir = os.path.join(output_dir, "preprocessed_data") 
    models_dir = os.path.join(output_dir, "models")
    figures_dir = os.path.join(output_dir, "figures")
    logs_dir = os.path.join(output_dir, "logs")

    for dir_path in [output_dir, data_dir, models_dir, figures_dir, logs_dir]:
        os.makedirs(dir_path, exist_ok=True)
    
    print(f"\n✓ Using existing preprocessed data")
    print(f"New model results will be saved to: {output_dir}")
    
    # Copy/link the data to the new experiment folder for completeness
    print("Copying preprocessed data to new experiment directory...")
    X_train.to_pickle(os.path.join(data_dir, "X_train.pkl"))
    X_dev.to_pickle(os.path.join(data_dir, "X_dev.pkl"))
    X_test.to_pickle(os.path.join(data_dir, "X_test.pkl"))
    Ys_train.to_pickle(os.path.join(data_dir, "Ys_train.pkl"))
    Ys_dev.to_pickle(os.path.join(data_dir, "Ys_dev.pkl"))
    Ys_test.to_pickle(os.path.join(data_dir, "Ys_test.pkl"))
    print("✓ Data copied to new experiment directory")


✓ Using existing preprocessed data
New model results will be saved to: ../results/baseline_experiment_20250621_170557
Copying preprocessed data to new experiment directory...
✓ Data copied to new experiment directory


In [6]:
# =============================================================================
# Section 5: Enhanced Model Selection & Training with Checkpoints and Outputs
# =============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix, classification_report
import json
import joblib

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print(f"\n--- Enhanced Model Selection for Target: '{TARGET_VARIABLE}' ---")
print(f"Results will be saved to: {output_dir}")

# Align target labels with the feature matrices
y_train_target = Ys_train.loc[X_train.index, TARGET_VARIABLE]
y_dev_target = Ys_dev.loc[X_dev.index, TARGET_VARIABLE]
y_test_target = Ys_test.loc[X_test.index, TARGET_VARIABLE]

print(f"\nTarget variable distribution:")
print(f"  Train: {y_train_target.value_counts().to_dict()}")
print(f"  Dev:   {y_dev_target.value_counts().to_dict()}")
print(f"  Test:  {y_test_target.value_counts().to_dict()}")

# Save target variables
y_train_target.to_pickle(os.path.join(data_dir, "y_train_target.pkl"))
y_dev_target.to_pickle(os.path.join(data_dir, "y_dev_target.pkl"))
y_test_target.to_pickle(os.path.join(data_dir, "y_test_target.pkl"))



--- Enhanced Model Selection for Target: 'mort_hosp' ---
Results will be saved to: ../results/baseline_experiment_20250621_170557

Target variable distribution:
  Train: {0.0: 14934, 1.0: 1826}
  Dev:   {0.0: 2171, 1.0: 223}
  Test:  {0.0: 4298, 1.0: 492}


In [ ]:
# =============================================================================
# Section 5.1: Advanced Hyperparameter Search with Checkpoints
# =============================================================================

import scipy.stats as ss
from sklearn.utils.fixes import loguniform

# Define utility classes for hyperparameter sampling
class Choice:
    """Discrete choice distribution"""
    def __init__(self, choices):
        self.choices = choices
    
    def rvs(self, size=1):
        if size == 1:
            return np.random.choice(self.choices)
        return np.random.choice(self.choices, size=size)

class DictDist:
    """Distribution over dictionaries of hyperparameters"""
    def __init__(self, param_dict):
        self.param_dict = param_dict
    
    def rvs(self, size=1):
        if size == 1:
            return {key: dist.rvs() for key, dist in self.param_dict.items()}
        return [{key: dist.rvs() for key, dist in self.param_dict.items()} for _ in range(size)]

# Define hyperparameter search parameters
N = 5

# XGBoost hyperparameter distribution
XGB_dist = DictDist({
    'max_depth': ss.randint(3, 12),                    # Random integer between 3-11
    'learning_rate': ss.loguniform(1e-3, 2e-1),       # Log-uniform between 0.001-0.2  
    'subsample': ss.uniform(0.5, 0.5),                # Uniform between 0.5-1.0
    'colsample_bytree': ss.uniform(0.5, 0.5),         # Uniform between 0.5-1.0
    'colsample_bylevel': ss.uniform(0.5, 0.5),        # Uniform between 0.5-1.0
    'min_child_weight': ss.randint(1, 10),            # Random integer between 1-9
    'gamma': ss.loguniform(1e-3, 10),                 # Log-uniform between 0.001-10
    'reg_alpha': ss.loguniform(1e-3, 100),            # L1 regularization
    'reg_lambda': ss.loguniform(1e-3, 100),           # L2 regularization
    'n_estimators': Choice([500, 750, 1000]),         # Fixed choices with early stopping
})

# Generate hyperparameter combinations
np.random.seed(SEED)
search_list = XGB_dist.rvs(N)

# Post-processing: ensure reasonable parameter combinations
for i, params in enumerate(search_list):
    # Ensure subsample + colsample_bytree isn't too restrictive
    if params['subsample'] < 0.7 and params['colsample_bytree'] < 0.7:
        params['subsample'] = min(1.0, params['subsample'] + 0.2)
    
    # Round continuous parameters to reasonable precision
    params['learning_rate'] = round(params['learning_rate'], 4)
    params['gamma'] = round(params['gamma'], 4)
    params['reg_alpha'] = round(params['reg_alpha'], 4)
    params['reg_lambda'] = round(params['reg_lambda'], 4)

# Optional: Add other model distributions following your pattern
LR_dist = DictDist({
    'C': Choice(np.geomspace(1e-3, 1e3, 100)),        # Geometric space like your example
    'penalty': Choice(['l1', 'l2']),
    'solver': Choice(['liblinear', 'lbfgs']),
    'max_iter': Choice([100, 500])
})

# Generate LR hyperparameters with post-processing rule
np.random.seed(SEED + 1)  # Different seed for variety
LR_hyperparams_list = LR_dist.rvs(N)
for i in range(N):
    # Apply the same constraint as your example
    if LR_hyperparams_list[i]['solver'] == 'lbfgs': 
        LR_hyperparams_list[i]['penalty'] = 'l2'

RF_dist = DictDist({
    'n_estimators': ss.randint(50, 500),
    'max_depth': ss.randint(2, 10),
    'min_samples_split': ss.randint(2, 75),
    'min_samples_leaf': ss.randint(1, 50),
})
np.random.seed(SEED + 2)  # Different seed for variety
RF_hyperparams_list = RF_dist.rvs(N)

print(f"Generated {N} hyperparameter combinations using scipy.stats distributions:")
print(f"\nXGBoost examples:")
for i, params in enumerate(search_list[:2], 1):  # Show first 2 examples
    print(f"  XGB {i}: {params}")

print(f"\nLogistic Regression examples:")
for i, params in enumerate(LR_hyperparams_list[:2], 1):
    print(f"  LR {i}: {params}")

print(f"\nRandom Forest examples:")  
for i, params in enumerate(RF_hyperparams_list[:2], 1):
    print(f"  RF {i}: {params}")

print(f"\nProceeding with XGBoost hyperparameter search...")

# Initialize tracking variables for all model types
all_results = {}
all_models = {}

# Define model configurations
model_configs = [
    {
        'name': 'XGBoost',
        'hyperparams': search_list,
        'model_class': xgb.XGBClassifier,
        'base_params': {'objective': 'binary:logistic', 'eval_metric': 'auc', 'use_label_encoder': False, 'seed': SEED}
    },
    {
        'name': 'LogisticRegression', 
        'hyperparams': LR_hyperparams_list,
        'model_class': None,  # Will import from sklearn
        'base_params': {'random_state': SEED}
    },
    {
        'name': 'RandomForest',
        'hyperparams': RF_hyperparams_list, 
        'model_class': None,  # Will import from sklearn
        'base_params': {'random_state': SEED}
    }
]

print(f"\nStarting comprehensive model comparison...")
print(f"Testing {N} hyperparameter combinations for each of 3 model types...")

# Import sklearn models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Update model classes
model_configs[1]['model_class'] = LogisticRegression
model_configs[2]['model_class'] = RandomForestClassifier

# Test each model type
for model_config in model_configs:
    model_name = model_config['name']
    hyperparams_list = model_config['hyperparams']
    ModelClass = model_config['model_class']
    base_params = model_config['base_params']
    
    print(f"\n{'='*60}")
    print(f"TESTING {model_name.upper()}")
    print(f"{'='*60}")
    
    model_results = []
    model_checkpoints = {}
    best_auc_model = -1
    best_params_model = None
    
    for i, params in enumerate(hyperparams_list):
        print(f"\n  [{i+1:2d}/{N}] {model_name}: {params}")
        
        try:
            # Create and train model
            if model_name == 'XGBoost':
                model = ModelClass(**base_params, **params)
                model.fit(
                    X_train, y_train_target,
                    eval_set=[(X_dev, y_dev_target)],
                    early_stopping_rounds=30,
                    verbose=True
                )
                auc_dev = model.best_score
                best_iteration = model.best_iteration
            else:
                model = ModelClass(**base_params, **params)
                model.fit(X_train, y_train_target)
                y_pred_dev = model.predict_proba(X_dev)[:, 1]
                auc_dev = roc_auc_score(y_dev_target, y_pred_dev)
                best_iteration = None
            
            # Save checkpoint
            checkpoint_path = os.path.join(models_dir, f"{model_name.lower()}_checkpoint_{i+1:02d}.pkl")
            joblib.dump(model, checkpoint_path)
            
            # Record results
            result = {
                'model_type': model_name,
                'iteration': i + 1,
                'params': params.copy(),
                'best_iteration': best_iteration,
                'auc_dev': auc_dev,
                'checkpoint_path': checkpoint_path
            }
            model_results.append(result)
            model_checkpoints[f'{model_name.lower()}_{i+1:02d}'] = model
            
            # Update best for this model type
            if auc_dev > best_auc_model:
                best_auc_model = auc_dev
                best_params_model = params.copy()
                if best_iteration is not None:
                    best_params_model['n_estimators'] = best_iteration
                print(f"    ★ NEW BEST {model_name}: AUC = {best_auc_model:.4f}")
            else:
                print(f"    AUC = {auc_dev:.4f}")
                
        except Exception as e:
            print(f"    ✗ ERROR: {str(e)}")
            continue
    
    # Store results for this model type
    all_results[model_name] = {
        'results': model_results,
        'best_auc': best_auc_model,
        'best_params': best_params_model,
        'checkpoints': model_checkpoints
    }
    
    print(f"\n✓ {model_name} search complete!")
    print(f"✓ Best {model_name} AUC: {best_auc_model:.4f}")
    print(f"✓ {len(model_results)} {model_name} models saved")

# Find overall best model across all types
print(f"\n{'='*80}")
print("OVERALL MODEL COMPARISON RESULTS")
print(f"{'='*80}")

best_overall_auc = -1
best_overall_model = None
best_overall_params = None

for model_name, results in all_results.items():
    auc = results['best_auc']
    print(f"{model_name:15s}: Best AUC = {auc:.4f}")
    
    if auc > best_overall_auc:
        best_overall_auc = auc
        best_overall_model = model_name
        best_overall_params = results['best_params']

print(f"\n🏆 OVERALL WINNER: {best_overall_model}")
print(f"🎯 Best AUC: {best_overall_auc:.4f}")
print(f"⚙️  Best parameters: {best_overall_params}")

# Set variables for compatibility with downstream code
best_auc = best_overall_auc
best_params = best_overall_params
search_results = []
for model_name, results in all_results.items():
    search_results.extend(results['results'])

print(f"\n✓ Comprehensive model comparison complete!")
print(f"✓ {len(search_results)} total models tested across 3 algorithms")
print(f"✓ All model checkpoints saved")


Generated 5 hyperparameter combinations using scipy.stats distributions:

XGBoost examples:
  XGB 1: {'max_depth': 8, 'learning_rate': 0.197, 'subsample': 0.9662786796693295, 'colsample_bytree': 0.5640622239646784, 'colsample_bylevel': 0.9995202576620723, 'min_child_weight': 1, 'gamma': 0.0023, 'reg_alpha': 0.0085, 'reg_lambda': 0.0534, 'n_estimators': 1000}
  XGB 2: {'max_depth': 5, 'learning_rate': 0.0886, 'subsample': 0.6566367584661376, 'colsample_bytree': 0.7622740797864357, 'colsample_bylevel': 0.7217264468897784, 'min_child_weight': 3, 'gamma': 4.5274, 'reg_alpha': 0.1932, 'reg_lambda': 0.1424, 'n_estimators': 1000}

Logistic Regression examples:
  LR 1: {'C': 0.26560877829466867, 'penalty': 'l2', 'solver': 'lbfgs', 'max_iter': 100}
  LR 2: {'C': 0.021544346900318846, 'penalty': 'l2', 'solver': 'liblinear', 'max_iter': 500}

Random Forest examples:
  RF 1: {'n_estimators': 412, 'max_depth': 2, 'min_samples_split': 5, 'min_samples_leaf': 9}
  RF 2: {'n_estimators': 306, 'max_dept

/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/xgboost/sklearn.py:797: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  UserWarning,


[0]	validation_0-auc:0.76149
[1]	validation_0-auc:0.79694
[2]	validation_0-auc:0.81527
[3]	validation_0-auc:0.81648
[4]	validation_0-auc:0.82743
[5]	validation_0-auc:0.83030
[6]	validation_0-auc:0.82666
[7]	validation_0-auc:0.83147
[8]	validation_0-auc:0.82882
[9]	validation_0-auc:0.83254
[10]	validation_0-auc:0.83745
[11]	validation_0-auc:0.84217
[12]	validation_0-auc:0.84296
[13]	validation_0-auc:0.84657
[14]	validation_0-auc:0.84864
[15]	validation_0-auc:0.84926
[16]	validation_0-auc:0.85168
[17]	validation_0-auc:0.85285
[18]	validation_0-auc:0.85389
[19]	validation_0-auc:0.85466
[20]	validation_0-auc:0.85658
[21]	validation_0-auc:0.85797
[22]	validation_0-auc:0.85837
[23]	validation_0-auc:0.85977
[24]	validation_0-auc:0.86045
[25]	validation_0-auc:0.86102
[26]	validation_0-auc:0.86086
[27]	validation_0-auc:0.86024
[28]	validation_0-auc:0.85965
[29]	validation_0-auc:0.86071
[30]	validation_0-auc:0.86206
[31]	validation_0-auc:0.86346
[32]	validation_0-auc:0.86424
[33]	validation_0-au

/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/sklearn/utils/validation.py:1692: FutureWarning: Feature names only support names that are all strings. Got feature names with dtypes: ['tuple']. An error will be raised in 1.2.
  FutureWarning,
/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/sklearn/linear_model/_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,
/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/sklearn/utils/validation.py:1692: FutureWarning: Feature names only support names that are all str

    ★ NEW BEST LogisticRegression: AUC = 0.7539

  [ 2/5] LogisticRegression: {'C': 0.021544346900318846, 'penalty': 'l2', 'solver': 'liblinear', 'max_iter': 500}


/Users/aaronge/miniconda3/envs/mimic_extract/lib/python3.7/site-packages/sklearn/utils/validation.py:1692: FutureWarning: Feature names only support names that are all strings. Got feature names with dtypes: ['tuple']. An error will be raised in 1.2.
  FutureWarning,


In [ ]:
# =============================================================================
# Section 5.2: Final Model Training and Comprehensive Evaluation
# =============================================================================

print(f"\n--- Final Model Training and Evaluation ---")
print(f"Using best model: {best_overall_model} (AUC: {best_overall_auc:.4f})")

# Train final model on combined train+dev data
print("Training final model on combined train+dev data...")
X_train_full = pd.concat([X_train, X_dev])
y_train_full_target = pd.concat([y_train_target, y_dev_target])

# Create final model based on the best performing algorithm
if best_overall_model == 'XGBoost':
    final_model = xgb.XGBClassifier(
        objective='binary:logistic', 
        use_label_encoder=False, 
        seed=SEED, 
        **best_params
    )
    final_model.fit(X_train_full, y_train_full_target)
elif best_overall_model == 'LogisticRegression':
    final_model = LogisticRegression(random_state=SEED, **best_params)
    final_model.fit(X_train_full, y_train_full_target)
elif best_overall_model == 'RandomForest':
    final_model = RandomForestClassifier(random_state=SEED, **best_params)
    final_model.fit(X_train_full, y_train_full_target)

# Save final model
final_model_path = os.path.join(models_dir, f"final_model_{best_overall_model.lower()}.pkl")
joblib.dump(final_model, final_model_path)
print(f"✓ Final {best_overall_model} model saved to: {final_model_path}")

# =============================================================================
# Section 5.3: Generate Predictions and Evaluate
# =============================================================================

print("\nGenerating predictions on all sets...")

# Generate predictions for all sets
sets_data = {
    'train': (X_train, y_train_target),
    'dev': (X_dev, y_dev_target),
    'test': (X_test, y_test_target),
    'train_full': (X_train_full, y_train_full_target)
}

predictions = {}
metrics = {}

for set_name, (X_set, y_set) in sets_data.items():
    # Generate predictions
    y_pred_proba = final_model.predict_proba(X_set)[:, 1]
    y_pred_binary = final_model.predict(X_set)
    
    # Calculate metrics
    from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score
    
    auroc = roc_auc_score(y_set, y_pred_proba)
    auprc = average_precision_score(y_set, y_pred_proba)
    accuracy = accuracy_score(y_set, y_pred_binary)
    f1 = f1_score(y_set, y_pred_binary)
    
    predictions[set_name] = {
        'y_true': y_set,
        'y_pred_proba': y_pred_proba,
        'y_pred_binary': y_pred_binary
    }
    
    metrics[set_name] = {
        'auroc': auroc,
        'auprc': auprc,
        'accuracy': accuracy,
        'f1_score': f1,
        'n_samples': len(y_set),
        'n_positive': int(y_set.sum()),
        'prevalence': y_set.mean()
    }
    
    print(f"  {set_name:10s}: AUROC={auroc:.4f}, AUPRC={auprc:.4f}, F1={f1:.4f}, Acc={accuracy:.4f}")

print(f"\n★ FINAL TEST PERFORMANCE ★")
print(f"  AUROC: {metrics['test']['auroc']:.4f}")
print(f"  AUPRC: {metrics['test']['auprc']:.4f}")
print(f"  F1 Score: {metrics['test']['f1_score']:.4f}")
print(f"  Accuracy: {metrics['test']['accuracy']:.4f}")


In [ ]:
# =============================================================================
# Section 5.4: Generate and Save All Figures and Results
# =============================================================================

print("\n--- Generating Figures and Saving Results ---")

# Create comprehensive figures
fig_size = (15, 12)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# 1. ROC Curves for all sets
plt.figure(figsize=(10, 8))
for i, (set_name, pred_data) in enumerate(predictions.items()):
    if set_name == 'train_full':  # Skip train_full to avoid clutter
        continue
    fpr, tpr, _ = roc_curve(pred_data['y_true'], pred_data['y_pred_proba'])
    auc = metrics[set_name]['auroc']
    plt.plot(fpr, tpr, color=colors[i], lw=2, 
             label=f'{set_name.capitalize()} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', alpha=0.5)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title(f'ROC Curves - {TARGET_VARIABLE.upper()} Prediction', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'roc_curves.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figures_dir, 'roc_curves.pdf'), bbox_inches='tight')
plt.close()

# 2. Precision-Recall Curves
plt.figure(figsize=(10, 8))
for i, (set_name, pred_data) in enumerate(predictions.items()):
    if set_name == 'train_full':
        continue
    precision, recall, _ = precision_recall_curve(pred_data['y_true'], pred_data['y_pred_proba'])
    auprc = metrics[set_name]['auprc']
    plt.plot(recall, precision, color=colors[i], lw=2,
             label=f'{set_name.capitalize()} (AUPRC = {auprc:.3f})')

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title(f'Precision-Recall Curves - {TARGET_VARIABLE.upper()} Prediction', fontsize=14, fontweight='bold')
plt.legend(loc="lower left", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'pr_curves.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figures_dir, 'pr_curves.pdf'), bbox_inches='tight')
plt.close()

# 3. Model Performance Comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
metrics_to_plot = ['auroc', 'auprc', 'f1_score', 'accuracy']
titles = ['AUROC', 'AUPRC', 'F1 Score', 'Accuracy']

for idx, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
    ax = axes[idx // 2, idx % 2]
    sets_to_plot = ['train', 'dev', 'test']
    values = [metrics[s][metric] for s in sets_to_plot]
    bars = ax.bar(sets_to_plot, values, color=colors[:3], alpha=0.7, edgecolor='black')
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Model Performance Metrics - {TARGET_VARIABLE.upper()} Prediction', 
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'performance_metrics.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figures_dir, 'performance_metrics.pdf'), bbox_inches='tight')
plt.close()

# 4. Model Comparison - Best Performance by Model Type
plt.figure(figsize=(10, 8))
model_names = list(all_results.keys())
model_aucs = [all_results[name]['best_auc'] for name in model_names]
model_colors = colors[:len(model_names)]

bars = plt.bar(model_names, model_aucs, color=model_colors, alpha=0.7, edgecolor='black', width=0.6)

# Add value labels on bars
for bar, auc in zip(bars, model_aucs):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{auc:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.xlabel('Model Type', fontsize=12)
plt.ylabel('Best Dev Set AUC', fontsize=12)
plt.title('Model Performance Comparison - Best AUC by Algorithm', fontsize=14, fontweight='bold')
plt.ylim(0, max(model_aucs) * 1.1)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'model_comparison.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figures_dir, 'model_comparison.pdf'), bbox_inches='tight')
plt.close()

# 5. Hyperparameter Search Progress by Model Type
plt.figure(figsize=(14, 8))
for i, (model_name, model_data) in enumerate(all_results.items()):
    model_results = model_data['results']
    if model_results:
        iterations = [r['iteration'] for r in model_results]
        aucs = [r['auc_dev'] for r in model_results]
        plt.plot(iterations, aucs, 'o-', linewidth=2, markersize=6, 
                color=colors[i], label=f'{model_name}', alpha=0.8)
        
        # Highlight best for this model
        best_result = max(model_results, key=lambda x: x['auc_dev'])
        plt.plot(best_result['iteration'], best_result['auc_dev'], 
                '*', markersize=12, color=colors[i], markeredgecolor='black', markeredgewidth=1)

plt.xlabel('Hyperparameter Search Iteration', fontsize=12)
plt.ylabel('Dev Set AUC', fontsize=12)
plt.title('Hyperparameter Search Progress by Model Type', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'hyperparameter_search_by_model.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figures_dir, 'hyperparameter_search_by_model.pdf'), bbox_inches='tight')
plt.close()

# 6. Hyperparameter Distribution Analysis by Model Type
# XGBoost Parameters
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

xgb_results = [r for r in search_results if r['model_type'] == 'XGBoost']
xgb_param_keys = ['learning_rate', 'max_depth', 'subsample', 'gamma', 'reg_alpha', 'reg_lambda']

for i, param in enumerate(xgb_param_keys):
    if i < len(axes):
        values = [result['params'][param] for result in xgb_results if param in result['params']]
        if values:
            axes[i].hist(values, bins=8, alpha=0.7, color=colors[0], edgecolor='black')
            axes[i].set_title(f'XGBoost: {param.replace("_", " ").title()}', fontweight='bold')
            axes[i].set_xlabel(param.replace("_", " ").title())
            axes[i].set_ylabel('Frequency')
            axes[i].grid(True, alpha=0.3)

plt.suptitle('XGBoost Hyperparameter Distributions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'xgboost_hyperparameter_distributions.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figures_dir, 'xgboost_hyperparameter_distributions.pdf'), bbox_inches='tight')
plt.close()

# Logistic Regression Parameters
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

lr_results = [r for r in search_results if r['model_type'] == 'LogisticRegression']
lr_param_keys = ['C', 'penalty', 'solver', 'max_iter']

for i, param in enumerate(lr_param_keys):
    if i < len(axes):
        values = [result['params'][param] for result in lr_results if param in result['params']]
        if values:
            if param == 'C':
                # Log scale for C parameter
                axes[i].hist(values, bins=8, alpha=0.7, color=colors[1], edgecolor='black')
                axes[i].set_xscale('log')
            elif param in ['penalty', 'solver']:
                # Categorical parameters
                unique_vals = list(set(values))
                counts = [values.count(val) for val in unique_vals]
                axes[i].bar(unique_vals, counts, alpha=0.7, color=colors[1], edgecolor='black')
            else:
                axes[i].hist(values, bins=8, alpha=0.7, color=colors[1], edgecolor='black')
            
            axes[i].set_title(f'LogReg: {param.replace("_", " ").title()}', fontweight='bold')
            axes[i].set_xlabel(param.replace("_", " ").title())
            axes[i].set_ylabel('Frequency')
            axes[i].grid(True, alpha=0.3)

plt.suptitle('Logistic Regression Hyperparameter Distributions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'logistic_regression_hyperparameter_distributions.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figures_dir, 'logistic_regression_hyperparameter_distributions.pdf'), bbox_inches='tight')
plt.close()

# Random Forest Parameters
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

rf_results = [r for r in search_results if r['model_type'] == 'RandomForest']
rf_param_keys = ['n_estimators', 'max_depth', 'min_samples_split', 'min_samples_leaf']

for i, param in enumerate(rf_param_keys):
    if i < len(axes):
        values = [result['params'][param] for result in rf_results if param in result['params']]
        if values:
            axes[i].hist(values, bins=8, alpha=0.7, color=colors[2], edgecolor='black')
            axes[i].set_title(f'RandomForest: {param.replace("_", " ").title()}', fontweight='bold')
            axes[i].set_xlabel(param.replace("_", " ").title())
            axes[i].set_ylabel('Frequency')
            axes[i].grid(True, alpha=0.3)

plt.suptitle('Random Forest Hyperparameter Distributions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'random_forest_hyperparameter_distributions.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figures_dir, 'random_forest_hyperparameter_distributions.pdf'), bbox_inches='tight')
plt.close()

# 7. Comprehensive Model Performance Matrix
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
metrics_to_plot = ['auroc', 'auprc', 'f1_score', 'accuracy']
titles = ['AUROC', 'AUPRC', 'F1 Score', 'Accuracy']

# Include model comparison in the performance metrics
sets_to_plot = ['train', 'dev', 'test']
for idx, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
    ax = axes[idx // 2, idx % 2]
    values = [metrics[s][metric] for s in sets_to_plot]
    bars = ax.bar(sets_to_plot, values, color=colors[:3], alpha=0.7, edgecolor='black')
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontweight='bold')
    
    ax.set_title(f'{title} - Final {best_overall_model} Model', fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Final Model Performance: {best_overall_model}', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, 'final_model_performance.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(figures_dir, 'final_model_performance.pdf'), bbox_inches='tight')
plt.close()

print("✓ All figures saved to:", figures_dir)
print("✓ Comprehensive multi-model visualization completed")
print(f"✓ Generated {7} figure sets covering all 3 model types")


In [ ]:
# =============================================================================
# Section 5.5: Save All Results and Create Summary
# =============================================================================

print("\n--- Saving All Results ---")

# Save hyperparameter search results
search_results_path = os.path.join(output_dir, "hyperparameter_search_results.json")
with open(search_results_path, 'w') as f:
    # Convert numpy types to Python types for JSON serialization
    search_results_json = []
    for result in search_results:
        result_copy = result.copy()
        result_copy['auc_dev'] = float(result_copy['auc_dev'])
        result_copy['best_iteration'] = int(result_copy['best_iteration'])
        search_results_json.append(result_copy)
    json.dump(search_results_json, f, indent=2)

# Save all metrics
metrics_path = os.path.join(output_dir, "final_metrics.json")
with open(metrics_path, 'w') as f:
    # Convert numpy types for JSON serialization
    metrics_json = {}
    for set_name, set_metrics in metrics.items():
        metrics_json[set_name] = {}
        for metric_name, value in set_metrics.items():
            if isinstance(value, (np.integer, np.floating)):
                metrics_json[set_name][metric_name] = float(value)
            else:
                metrics_json[set_name][metric_name] = value
    json.dump(metrics_json, f, indent=2)

# Save predictions
predictions_path = os.path.join(output_dir, "predictions.pkl")
with open(predictions_path, 'wb') as f:
    pickle.dump(predictions, f)

# Create comprehensive summary report
summary_path = os.path.join(output_dir, "experiment_summary.txt")
with open(summary_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("BASELINE MODEL EXPERIMENT SUMMARY\n")
    f.write("="*80 + "\n")
    f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Target Variable: {TARGET_VARIABLE}\n")
    f.write(f"Window Size: {WINDOW_SIZE} hours\n")
    f.write(f"Gap Time: {GAP_TIME} hours\n")
    f.write(f"Random Seed: {SEED}\n\n")
    
    f.write("DATA SPLITS:\n")
    f.write("-"*40 + "\n")
    f.write(f"Train subjects: {len(train_subj):,}\n")
    f.write(f"Dev subjects:   {len(dev_subj):,}\n") 
    f.write(f"Test subjects:  {len(test_subj):,}\n")
    f.write(f"Total subjects: {len(train_subj) + len(dev_subj) + len(test_subj):,}\n\n")
    
    f.write("FEATURE DIMENSIONS:\n")
    f.write("-"*40 + "\n")
    f.write(f"Train features: {X_train.shape}\n")
    f.write(f"Dev features:   {X_dev.shape}\n")
    f.write(f"Test features:  {X_test.shape}\n\n")
    
    f.write("TARGET VARIABLE DISTRIBUTION:\n")
    f.write("-"*40 + "\n")
    for set_name in ['train', 'dev', 'test']:
        pos = metrics[set_name]['n_positive']
        total = metrics[set_name]['n_samples']
        prev = metrics[set_name]['prevalence']
        f.write(f"{set_name.capitalize():5s}: {pos:,}/{total:,} ({prev:.1%} positive)\n")
    f.write("\n")
    
    f.write("COMPREHENSIVE MODEL COMPARISON:\n")
    f.write("-"*40 + "\n")
    f.write(f"Models tested: XGBoost, Logistic Regression, Random Forest\n")
    f.write(f"Search method: scipy.stats distributions\n")
    f.write(f"Total model evaluations: {len(search_results)}\n")
    f.write(f"Hyperparameter combinations per model: {N}\n\n")
    
    f.write("BEST MODEL BY TYPE:\n")
    for model_name, results in all_results.items():
        f.write(f"  {model_name:15s}: {results['best_auc']:.4f}\n")
    
    f.write(f"\nOVERALL WINNER: {best_overall_model}\n")
    f.write(f"Best dev AUC: {best_auc:.4f}\n")
    f.write(f"Best parameters: {best_params}\n\n")
    
    f.write("PARAMETER DISTRIBUTIONS USED:\n")
    f.write(f"XGBoost:\n")
    f.write(f"  - learning_rate: log-uniform(1e-3, 2e-1)\n")
    f.write(f"  - max_depth: randint(3, 12)\n")
    f.write(f"  - subsample: uniform(0.5, 1.0)\n")
    f.write(f"  - gamma: log-uniform(1e-3, 10)\n")
    f.write(f"  - reg_alpha: log-uniform(1e-3, 100)\n")
    f.write(f"  - reg_lambda: log-uniform(1e-3, 100)\n")
    f.write(f"Logistic Regression:\n")
    f.write(f"  - C: geomspace(1e-3, 1e3, 100)\n")
    f.write(f"  - penalty: choice(['l1', 'l2'])\n")
    f.write(f"  - solver: choice(['liblinear', 'lbfgs'])\n")
    f.write(f"Random Forest:\n")
    f.write(f"  - n_estimators: randint(50, 500)\n")
    f.write(f"  - max_depth: randint(2, 10)\n")
    f.write(f"  - min_samples_split: randint(2, 75)\n")
    f.write(f"  - min_samples_leaf: randint(1, 50)\n\n")
    
    f.write("FINAL MODEL PERFORMANCE:\n")
    f.write("-"*40 + "\n")
    for set_name in ['train', 'dev', 'test']:
        m = metrics[set_name]
        f.write(f"{set_name.upper():5s} SET:\n")
        f.write(f"  AUROC:    {m['auroc']:.4f}\n")
        f.write(f"  AUPRC:    {m['auprc']:.4f}\n")
        f.write(f"  F1 Score: {m['f1_score']:.4f}\n")
        f.write(f"  Accuracy: {m['accuracy']:.4f}\n\n")
    
    f.write("FILES GENERATED:\n")
    f.write("-"*40 + "\n")
    f.write(f"Output directory: {output_dir}\n")
    f.write(f"├─ preprocessed_data/     # All preprocessed datasets\n")
    f.write(f"├─ models/               # All model checkpoints + final model\n")
    f.write(f"│  ├─ xgboost_checkpoint_*.pkl    # XGBoost model checkpoints\n")
    f.write(f"│  ├─ logisticregression_checkpoint_*.pkl # LogReg checkpoints\n")
    f.write(f"│  ├─ randomforest_checkpoint_*.pkl # RandomForest checkpoints\n")
    f.write(f"│  └─ final_model_{best_overall_model.lower() if best_overall_model else 'unknown'}.pkl # Best final model\n")
    f.write(f"├─ figures/              # Comprehensive visualization suite\n")
    f.write(f"│  ├─ roc_curves.png/pdf          # ROC curve comparison\n")
    f.write(f"│  ├─ pr_curves.png/pdf           # Precision-recall curves\n")
    f.write(f"│  ├─ model_comparison.png/pdf    # Model type comparison\n")
    f.write(f"│  ├─ hyperparameter_search_by_model.png/pdf # Search progress by model\n")
    f.write(f"│  ├─ xgboost_hyperparameter_distributions.png/pdf # XGBoost params\n")
    f.write(f"│  ├─ logistic_regression_hyperparameter_distributions.png/pdf # LogReg params\n")
    f.write(f"│  ├─ random_forest_hyperparameter_distributions.png/pdf # RF params\n")
    f.write(f"│  └─ final_model_performance.png/pdf # Final model metrics\n")
    f.write(f"├─ logs/                 # Log files (if any)\n")
    f.write(f"├─ experiment_config.pkl # Experiment configuration\n")
    f.write(f"├─ hyperparameter_search_results.json # Multi-model search results\n")
    f.write(f"├─ final_metrics.json    # All performance metrics\n")
    f.write(f"├─ predictions.pkl       # All model predictions\n")
    f.write(f"└─ experiment_summary.txt # This summary file\n")

print("✓ Results summary saved to:", summary_path)
print("✓ Hyperparameter search results saved to:", search_results_path)
print("✓ Final metrics saved to:", metrics_path)
print("✓ Predictions saved to:", predictions_path)

print(f"\n" + "="*80)
print("COMPREHENSIVE MULTI-MODEL EXPERIMENT COMPLETED!")
print("="*80)
print(f"📁 All results saved to: {output_dir}")
print(f"🎯 Final Test AUROC: {metrics['test']['auroc']:.4f}")
print(f"🏆 Best Model: {best_overall_model} (AUC: {best_auc:.4f})")
print(f"📊 {len(search_results)} total model evaluations across 3 algorithms")
print(f"📈 8 comprehensive figure sets generated")
print(f"🔬 Advanced scipy.stats hyperparameter search completed")
print(f"📊 Model-specific hyperparameter distribution analysis")
print(f"🔄 Multi-algorithm comparison: XGBoost vs LogReg vs RandomForest")
print(f"💾 All data, models, and results preserved for reproducibility")
print(f"⚡ Smart data loading: skip preprocessing on future runs")
print("="*80)
